# 08 — Final Visualization
Generates all report-quality plots for the 3-page submission report.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import lightkurve as lk
import os, glob, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.labelsize'   : 11,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'figure.dpi'       : 150,
    'axes.grid'        : True,
    'grid.alpha'       : 0.25,
    'grid.linestyle'   : '--',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.facecolor'   : '#fafafa',
    'figure.facecolor' : 'white',
    'savefig.facecolor': 'white',
})

PROCESSED_DIR = '../data/processed/'
PLOTS_DIR     = '../outputs/plots/'
EXCLUDE_TICS  = ['261203535']
os.makedirs(PLOTS_DIR, exist_ok=True)

LABEL_COLORS = {
    'planet'           : '#27ae60',
    'planet_candidate' : '#2ecc71',
    'eclipsing_binary' : '#e74c3c',
    'false_positive'   : '#e67e22',
    'noise'            : '#95a5a6',
    'unknown'          : '#f39c12',
}

print('Settings ready!')

## 2. Load All Results

In [ ]:
final_df  = pd.read_csv('../outputs/final_classification.csv')
bls_df    = pd.read_csv('../outputs/bls_all_results.csv')
params_df = pd.read_csv('../outputs/transit_params.csv') if os.path.exists('../outputs/transit_params.csv') else pd.DataFrame()
planet_df = pd.read_csv('../outputs/planet_parameters.csv') if os.path.exists('../outputs/planet_parameters.csv') else pd.DataFrame()

for df in [final_df, bls_df]:
    df['tic_id'] = df['tic_id'].astype(str).str.replace('.0', '', regex=False)
if not params_df.empty:
    params_df['tic_id'] = params_df['tic_id'].astype(str).str.replace('.0', '', regex=False)
if not planet_df.empty:
    planet_df['tic_id'] = planet_df['tic_id'].astype(str).str.replace('.0', '', regex=False)

final_df = final_df[~final_df['tic_id'].isin(EXCLUDE_TICS)].reset_index(drop=True)

# Use raw SNR for display (not normalised)
if 'bls_snr_raw' in final_df.columns:
    final_df['snr_display'] = final_df['bls_snr_raw']
else:
    final_df['snr_display'] = final_df['snr']

print(f'Stars : {len(final_df)}')
print(final_df[['tic_id', 'ml_label', 'snr_display', 'depth_ppm']].to_string())

## 3. PLOT 1 — Pipeline Overview Figure
Shows one example star through the full pipeline: raw → cleaned → phase-folded

In [ ]:
def bin_phase(phase_s, flux_s, n_bins=75):
    bins     = np.linspace(-0.5, 0.5, n_bins + 1)
    mid      = (bins[:-1] + bins[1:]) / 2
    medians, stds = [], []
    for i in range(n_bins):
        m = (phase_s >= bins[i]) & (phase_s < bins[i+1])
        if m.sum() > 0:
            medians.append(np.nanmedian(flux_s[m]))
            stds.append(np.nanstd(flux_s[m]) / np.sqrt(m.sum()))
        else:
            medians.append(np.nan)
            stds.append(np.nan)
    return mid, np.array(medians), np.array(stds)


# Pick best demo star — highest raw SNR, valid depth
best_row = final_df.sort_values('snr_display', ascending=False).iloc[0]
tic_id   = str(best_row['tic_id']).replace('.0', '')
period   = float(best_row.get('period_days', 5.0))
t0       = float(best_row['transit_time'])
label    = best_row['ml_label']
conf     = float(best_row['ml_confidence'])
color    = LABEL_COLORS.get(label, '#3498db')

raw_fits = glob.glob(f'../data/raw/TIC_{tic_id}.fits') or \
           glob.glob(f'../data/labeled/TIC_{tic_id}*.fits')

proc_csv = os.path.join(PROCESSED_DIR, f'TIC_{tic_id}.csv')
df_proc  = pd.read_csv(proc_csv)
time_p   = df_proc['time'].values
flux_p   = df_proc['flux'].values
mk       = np.isfinite(time_p) & np.isfinite(flux_p)
time_p, flux_p = time_p[mk], flux_p[mk]

phase    = ((time_p - t0) % period) / period
phase[phase > 0.5] -= 1.0
sidx     = np.argsort(phase)
phase_s, flux_s = phase[sidx], flux_p[sidx]
bin_mid, bin_med, bin_err = bin_phase(phase_s, flux_s)

fig = plt.figure(figsize=(18, 13))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35,
                         height_ratios=[1.2, 1, 1.1])

# Panel A — Raw light curve
ax1 = fig.add_subplot(gs[0, :])
if raw_fits:
    try:
        lc_raw = lk.read(raw_fits[0])
        tr     = lc_raw.time.value
        fr     = lc_raw.flux.value
        mk_r   = np.isfinite(fr)
        ax1.plot(tr[mk_r], fr[mk_r], '.', color='#7f8c8d', markersize=1, alpha=0.45)
        ax1.set_ylabel('Raw Flux (e⁻/s)')
    except:
        ax1.plot(time_p, flux_p, '.', color='#7f8c8d', markersize=1, alpha=0.45)
        ax1.set_ylabel('Normalized Flux')
else:
    ax1.plot(time_p, flux_p, '.', color='#7f8c8d', markersize=1, alpha=0.45)
    ax1.set_ylabel('Normalized Flux')
ax1.set_xlabel('Time (BTJD days)')
ax1.set_title(f'(A) Raw TESS Photometry — TIC {tic_id}  [{label.replace("_"," ").title()}]',
              fontweight='bold')

# Panel B — Cleaned / normalized
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(time_p, flux_p, '.', color='#2980b9', markersize=1, alpha=0.4)
# Mark transit windows
n_tr = int((time_p[-1] - time_p[0]) / period)
for i in range(n_tr + 1):
    tc   = t0 + i * period
    dur  = float(best_row.get('duration_days', 0.1))
    ax2.axvspan(tc - dur/2, tc + dur/2, alpha=0.15, color=color, lw=0)
ax2.set_xlabel('Time (BTJD days)')
ax2.set_ylabel('Normalized Flux')
ax2.set_title('(B) After Savitzky-Golay Detrending + Sigma Clipping  '
              '(shaded = transit windows)', fontweight='bold')

# Panel C — Full phase fold
ax3 = fig.add_subplot(gs[2, 0])
ax3.plot(phase_s, flux_s, '.', color='#bdc3c7', markersize=1.2, alpha=0.25, zorder=1)
ok = ~np.isnan(bin_med)
ax3.errorbar(bin_mid[ok], bin_med[ok], yerr=bin_err[ok],
             fmt='o', color=color, markersize=4, lw=1.5,
             elinewidth=1, capsize=2, zorder=5, label='Binned median ± stderr')
ax3.axhline(1.0, color='k', lw=0.8, linestyle='--', alpha=0.4)
dur_ph = float(best_row.get('duration_days', 0.1)) / period
ax3.axvspan(-dur_ph/2, dur_ph/2, alpha=0.1, color=color, label='Transit window')
ax3.set_xlim(-0.5, 0.5)
ax3.set_xlabel('Orbital Phase')
ax3.set_ylabel('Normalized Flux')
ax3.set_title(f'(C) Phase-Folded  P = {period:.4f} days', fontweight='bold')
ax3.legend(fontsize=8)

# Panel D — Zoomed transit
ax4 = fig.add_subplot(gs[2, 1])
zoom = min(0.12, dur_ph * 4)
zm   = np.abs(phase_s) < zoom
bz   = np.abs(bin_mid) < zoom
ax4.plot(phase_s[zm] * period * 24, flux_s[zm],
         '.', color='#bdc3c7', markersize=2.5, alpha=0.4)
ax4.errorbar(bin_mid[bz & ok] * period * 24, bin_med[bz & ok],
             yerr=bin_err[bz & ok],
             fmt='o', color=color, markersize=6,
             elinewidth=1.5, capsize=3, zorder=5)
ax4.axhline(1.0, color='k', lw=0.8, linestyle='--', alpha=0.4)
ax4.set_xlabel('Time from Transit Center (hours)')
ax4.set_ylabel('Normalized Flux')
depth_show = float(best_row.get('depth_ppm', 0))
ax4.set_title(
    f'(D) Transit Zoom\n'
    f'{label.replace("_"," ").title()}  •  {conf:.0%} confidence  '
    f'•  depth ≈ {depth_show:.0f} ppm',
    fontweight='bold', color=color
)

plt.suptitle(
    f'Figure 1 — Full Pipeline Walk-Through: TIC {tic_id}\n'
    f'Raw Photometry → Detrending → Phase Folding → Transit Detection',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig(os.path.join(PLOTS_DIR, 'FIGURE1_pipeline_overview.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('FIGURE 1 saved!')

## 4. PLOT 2 — Classification Results Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 13))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# ── A: Pie chart with counts ────────────────────────────────────────
ax_pie = fig.add_subplot(gs[0, 0])
cnt    = final_df['ml_label'].value_counts()
colors_pie = [LABEL_COLORS.get(l, '#95a5a6') for l in cnt.index]
wedges, texts, autotexts = ax_pie.pie(
    cnt.values, labels=None, colors=colors_pie,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12, 'fontweight': 'bold'}
)
ax_pie.legend(
    wedges,
    [f'{l.replace("_"," ").title()}  (n={v})' for l, v in cnt.items()],
    loc='lower center', bbox_to_anchor=(0.5, -0.18),
    fontsize=9, framealpha=0.9
)
ax_pie.set_title('(A) Signal Classification\nBreakdown', fontweight='bold')

# ── B: SNR bar chart ────────────────────────────────────────────────
ax_snr = fig.add_subplot(gs[0, 1])
sorted_df  = final_df.sort_values('snr_display', ascending=False).reset_index(drop=True)
bar_colors = [LABEL_COLORS.get(l, '#95a5a6') for l in sorted_df['ml_label']]
bars = ax_snr.bar(range(len(sorted_df)), sorted_df['snr_display'],
                   color=bar_colors, edgecolor='white', linewidth=0.5, width=0.75)
ax_snr.axhline(7, color='#e74c3c', linestyle='--', lw=1.8, label='SNR = 7 (strong)')
ax_snr.axhline(5, color='#f39c12', linestyle=':', lw=1.5, label='SNR = 5 (marginal)')
ax_snr.set_xticks(range(len(sorted_df)))
ax_snr.set_xticklabels(
    [t[-6:] for t in sorted_df['tic_id']], rotation=55, ha='right', fontsize=8
)
ax_snr.set_ylabel('Signal-to-Noise Ratio')
ax_snr.set_title('(B) SNR per Star\n(sorted descending)', fontweight='bold')
patches = [mpatches.Patch(color=v, label=k.replace('_',' ').title())
           for k, v in LABEL_COLORS.items() if k in final_df['ml_label'].values]
ax_snr.legend(handles=patches + [
    plt.Line2D([0],[0], color='#e74c3c', linestyle='--', label='SNR=7'),
    plt.Line2D([0],[0], color='#f39c12', linestyle=':', label='SNR=5')
], fontsize=8, loc='upper right')

# ── C: Period vs Depth — detection space ────────────────────────────
ax_pd = fig.add_subplot(gs[0, 2])
for lbl, grp in final_df.groupby('ml_label'):
    col  = LABEL_COLORS.get(lbl, '#3498db')
    size = np.clip(grp['snr_display'] * 8, 30, 400)
    ax_pd.scatter(grp['period_days'], grp['depth_ppm'],
                  c=col, s=size, edgecolors='k', linewidths=0.5,
                  alpha=0.85, label=lbl.replace('_',' ').title(), zorder=5)
    for _, row in grp.iterrows():
        ax_pd.annotate(row['tic_id'][-5:],
                       (row['period_days'], row['depth_ppm']),
                       textcoords='offset points', xytext=(4, 4),
                       fontsize=6.5, color=col, alpha=0.8)
ax_pd.set_xlabel('Orbital Period (days)')
ax_pd.set_ylabel('Transit Depth (ppm)')
ax_pd.set_yscale('log')
ax_pd.set_title('(C) Period–Depth Detection Space\n(size = SNR)', fontweight='bold')
ax_pd.legend(fontsize=8, loc='upper right')
# Planet / EB reference lines
ax_pd.axhline(10_000, color='gray', lw=0.7, linestyle='--', alpha=0.5)
ax_pd.text(0.5, 10_200, 'Typical hot Jupiter (~10,000 ppm)', fontsize=7, color='gray')

# ── D: Confidence histogram ──────────────────────────────────────────
ax_ch = fig.add_subplot(gs[1, 0])
for lbl, grp in final_df.groupby('ml_label'):
    ax_ch.hist(grp['ml_confidence'] * 100, bins=8, alpha=0.75,
               color=LABEL_COLORS.get(lbl, '#95a5a6'),
               edgecolor='white', label=lbl.replace('_',' ').title())
ax_ch.axvline(70, color='#e74c3c', linestyle='--', lw=1.5, label='70% threshold')
ax_ch.set_xlabel('Classification Confidence (%)')
ax_ch.set_ylabel('Count')
ax_ch.set_title('(D) Confidence Distribution\nby Class', fontweight='bold')
ax_ch.legend(fontsize=8)

# ── E: Duration vs Period (Kepler / planet geometry) ─────────────────
ax_dp = fig.add_subplot(gs[1, 1])
if 'duration_hours' in final_df.columns:
    for lbl, grp in final_df.groupby('ml_label'):
        col = LABEL_COLORS.get(lbl, '#3498db')
        ax_dp.scatter(grp['period_days'], grp['duration_hours'],
                      c=col, s=70, edgecolors='k', linewidths=0.5,
                      alpha=0.85, label=lbl.replace('_',' ').title())
    # Theoretical duration lines for a = 0.05, 0.1 AU (solar-type star)
    P_arr = np.linspace(0.5, 13, 200)
    for a_au, ls in [(0.05, '--'), (0.1, ':')]:
        T_dur = (P_arr / np.pi) * np.arcsin(0.00465 / a_au) * 24
        ax_dp.plot(P_arr, T_dur, color='gray', lw=1, linestyle=ls,
                   label=f'a={a_au} AU (1R☉ star)', alpha=0.7)
    ax_dp.set_xlabel('Orbital Period (days)')
    ax_dp.set_ylabel('Transit Duration (hours)')
    ax_dp.set_title('(E) Transit Duration vs Period\n(lines = theoretical geometry)', fontweight='bold')
    ax_dp.legend(fontsize=7)

# ── F: Summary stats text panel ──────────────────────────────────────
ax_txt = fig.add_subplot(gs[1, 2])
ax_txt.axis('off')
label_cnt = final_df['ml_label'].value_counts()
lines = ['DETECTION SUMMARY', '─' * 30]
lines.append(f'Total stars analysed : {len(final_df)}')
for lbl, cnt_v in label_cnt.items():
    lines.append(f'  {lbl.replace("_"," ").title():22s}: {cnt_v}')
lines += ['', 'SNR STATISTICS']
lines += ['─' * 30]
lines += [f'  Max SNR : {final_df["snr_display"].max():.1f}']
lines += [f'  Med SNR : {final_df["snr_display"].median():.1f}']
lines += [f'  Stars with SNR≥7 : {(final_df["snr_display"]>=7).sum()}']
lines += ['', 'CONFIDENCE']
lines += ['─' * 30]
lines += [f'  Mean confidence : {final_df["ml_confidence"].mean()*100:.0f}%']
lines += [f'  >70% confident  : {(final_df["ml_confidence"]>0.7).sum()} stars']
ax_txt.text(0.05, 0.97, '\n'.join(lines),
            transform=ax_txt.transAxes,
            fontsize=10, verticalalignment='top',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#eaf4fb', alpha=0.9, pad=0.8))
ax_txt.set_title('(F) Pipeline Statistics', fontweight='bold')

plt.suptitle('Figure 2 — Classification Results Dashboard\nAll Science Targets',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(os.path.join(PLOTS_DIR, 'FIGURE2_classification_dashboard.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('FIGURE 2 saved!')

## 5. PLOT 3 — Phase-Folded Gallery (top 6 stars by SNR)

In [ ]:
top6 = final_df.nlargest(6, 'snr_display').reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(19, 11))
axes = axes.flatten()

for idx, (_, row) in enumerate(top6.iterrows()):
    ax     = axes[idx]
    tic_id = str(row['tic_id']).replace('.0', '')
    label  = row['ml_label']
    conf   = float(row['ml_confidence'])
    period = float(row.get('period_days', 5.0))
    t0     = float(row['transit_time'])
    snr    = float(row.get('snr_display', row.get('snr', 0)))
    depth  = float(row.get('depth_ppm', 0))
    dur_d  = float(row.get('duration_days', 0.1))
    color  = LABEL_COLORS.get(label, '#3498db')

    try:
        df_lc = pd.read_csv(os.path.join(PROCESSED_DIR, f'TIC_{tic_id}.csv'))
        time  = df_lc['time'].values
        flux  = df_lc['flux'].values
        mask  = np.isfinite(time) & np.isfinite(flux)
        time, flux = time[mask], flux[mask]

        phase = ((time - t0) % period) / period
        phase[phase > 0.5] -= 1.0
        sidx  = np.argsort(phase)
        ps, fs = phase[sidx], flux[sidx]

        bm, bmed, berr = bin_phase(ps, fs, n_bins=60)
        ok = ~np.isnan(bmed)

        # Raw points (downsampled for speed)
        step = max(1, len(ps) // 3000)
        ax.plot(ps[::step], fs[::step], '.', color='#bdc3c7',
                markersize=1.2, alpha=0.25, zorder=1)

        # Binned with error bars
        ax.errorbar(bm[ok], bmed[ok], yerr=berr[ok],
                    fmt='o', color=color, markersize=4.5, lw=1.5,
                    elinewidth=1, capsize=2, zorder=5, label='Binned')

        # Shade transit region
        dur_ph = dur_d / period
        ax.axvspan(-dur_ph/2, dur_ph/2, alpha=0.08, color=color, lw=0)
        ax.axhline(1.0, color='k', lw=0.7, linestyle='--', alpha=0.35)

        ax.set_xlim(-0.5, 0.5)
        ax.set_xlabel('Orbital Phase', fontsize=9)
        ax.set_ylabel('Normalized Flux', fontsize=9)

        # Color-coded title
        title_color = color
        ax.set_title(
            f'TIC {tic_id}',
            fontsize=10, fontweight='bold', color=title_color, pad=3
        )

        # Info box (bottom-right corner)
        fap_str = ''
        if 'tls_fap' in row and pd.notna(row['tls_fap']):
            fap_str = f'\nFAP = {float(row["tls_fap"]):.4f}'
        box_text = (
            f'{label.replace("_"," ").title()}\n'
            f'Confidence: {conf:.0%}\n'
            f'P = {period:.3f} d\n'
            f'Depth = {depth:.0f} ppm\n'
            f'SNR = {snr:.1f}'
            f'{fap_str}'
        )
        ax.text(0.98, 0.05, box_text,
                transform=ax.transAxes, fontsize=7.5,
                va='bottom', ha='right',
                bbox=dict(boxstyle='round', facecolor='white',
                          edgecolor=color, alpha=0.9, linewidth=1.2))

    except Exception as e:
        ax.text(0.5, 0.5, f'TIC {tic_id}\nError: {str(e)[:45]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=8)

# Shared legend
patches_leg = [mpatches.Patch(color=v, label=k.replace('_', ' ').title())
               for k, v in LABEL_COLORS.items()]
fig.legend(handles=patches_leg, loc='lower center', ncol=len(patches_leg),
           fontsize=10, bbox_to_anchor=(0.5, -0.02),
           framealpha=0.95, edgecolor='#bdc3c7')

plt.suptitle(
    'Figure 3 — Phase-Folded Light Curves: Top 6 Stars by SNR\n'
    'Binned median ± standard error  |  Shaded region = transit window',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'FIGURE3_top6_phasefold.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('FIGURE 3 saved!')

## 6. PLOT 4 — Parameter Estimation Table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: Styled results table ───────────────────────────────────────
ax_t = axes[0]
ax_t.axis('off')

table_df = final_df[['tic_id', 'ml_label', 'ml_confidence',
                      'period_days', 'depth_ppm', 'snr_display']].copy()
if 'duration_hours' in final_df.columns:
    table_df['duration_hours'] = final_df['duration_hours'].round(2)
table_df = table_df.sort_values('snr_display', ascending=False).reset_index(drop=True)

col_labels = ['TIC ID', 'Label', 'Confidence', 'Period\n(days)',
              'Depth\n(ppm)', 'Duration\n(hr)', 'SNR']
cell_text, row_colors = [], []
for _, r in table_df.iterrows():
    lbl   = r['ml_label']
    base  = LABEL_COLORS.get(lbl, '#ecf0f1')
    cell_text.append([
        r['tic_id'],
        lbl.replace('_', ' ').title(),
        f"{r['ml_confidence']:.0%}",
        f"{r['period_days']:.3f}",
        f"{r['depth_ppm']:.0f}",
        f"{r.get('duration_hours', 0):.2f}" if 'duration_hours' in r else '—',
        f"{r['snr_display']:.1f}"
    ])
    row_colors.append([base + '55'] * 7)

tbl = ax_t.table(cellText=cell_text, colLabels=col_labels,
                 cellLoc='center', loc='center', cellColours=row_colors)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
tbl.scale(1, 1.7)
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')
ax_t.set_title('(A) Complete Results Table — All Science Targets\n'
               '(sorted by SNR, colour = class)',
               fontsize=11, fontweight='bold', pad=18)

# ── Right: Planet parameter dot plot (if data available) ─────────────
ax_r = axes[1]
if not planet_df.empty and 'radius_earth' in planet_df.columns:
    pf = planet_df.copy()
    pf['tic_id'] = pf['tic_id'].astype(str).str.replace('.0', '', regex=False)
    pf = pf[pf['radius_earth'] < 30].reset_index(drop=True)

    y_pos   = np.arange(len(pf))
    r_earth = pf['radius_earth'].values
    teq     = pf.get('teq_kelvin', pd.Series([np.nan]*len(pf))).values
    colors_dot = plt.cm.RdYlBu_r(
        np.clip((teq - 300) / 2500, 0, 1)
    ) if not np.all(np.isnan(teq)) else ['#3498db'] * len(pf)

    sc = ax_r.scatter(r_earth, y_pos, c=colors_dot, s=120,
                      edgecolors='k', linewidths=0.7, zorder=5)
    ax_r.set_yticks(y_pos)
    ax_r.set_yticklabels([f"TIC {t[-6:]}" for t in pf['tic_id']], fontsize=9)
    ax_r.set_xlabel('Planet Radius (R⊕)', fontsize=11)
    ax_r.axvline(1.0,  color='blue',   lw=1, linestyle='--', alpha=0.5, label='Earth (1 R⊕)')
    ax_r.axvline(3.9,  color='orange', lw=1, linestyle='--', alpha=0.5, label='Neptune (3.9 R⊕)')
    ax_r.axvline(11.2, color='red',    lw=1, linestyle='--', alpha=0.5, label='Jupiter (11.2 R⊕)')
    ax_r.legend(fontsize=8)

    # Colorbar for equilibrium temp
    if not np.all(np.isnan(teq)):
        sm = plt.cm.ScalarMappable(
            cmap='RdYlBu_r',
            norm=plt.Normalize(vmin=300, vmax=2800)
        )
        sm.set_array([])
        cb = plt.colorbar(sm, ax=ax_r, pad=0.02)
        cb.set_label('Equilibrium Temperature (K)', fontsize=9)

    ax_r.set_title('(B) Estimated Planet Radii\n(colour = equilibrium temperature)',
                   fontsize=11, fontweight='bold')
    ax_r.set_xlim(0, max(r_earth.max() * 1.15, 13))
    ax_r.invert_yaxis()
else:
    ax_r.text(0.5, 0.5,
              'Planet physical parameters\nnot yet computed.\nRun notebook 07 first.',
              ha='center', va='center', transform=ax_r.transAxes, fontsize=12)
    ax_r.set_title('(B) Planet Parameters', fontsize=11, fontweight='bold')

plt.suptitle('Figure 4 — Parameter Estimation Results',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'FIGURE4_parameters.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('FIGURE 4 saved!')

In [ ]:
# ── FIGURE 5: Robovetter-style Vetting Scorecard ─────────────────────
metrics_map = {
    'SNR Score'  : lambda r: min(r.get('snr_display', r.get('snr', 0)) / 15, 1.0),
    'Depth OK'   : lambda r: 1.0 if 100 < r.get('depth_ppm', 0) < 200_000 else 0.2,
    'FAP Score'  : lambda r: max(0, 1 - r.get('tls_fap', 1.0)) if pd.notna(r.get('tls_fap', np.nan)) else 0.5,
    'BIC Score'  : lambda r: min(max(r.get('delta_bic', 0), 0) / 200, 1.0),
    'Conf Score' : lambda r: float(r.get('ml_confidence', 0)),
    'Not EB'     : lambda r: 0.1 if r.get('ml_label', '') == 'eclipsing_binary'
                              else (0.5 if r.get('ml_label', '') == 'false_positive'
                              else 1.0),
}

sorted_vdf  = final_df.sort_values('snr_display', ascending=False).reset_index(drop=True)
score_matrix = []
for _, row in sorted_vdf.iterrows():
    score_matrix.append([fn(row) for fn in metrics_map.values()])

score_arr  = np.array(score_matrix, dtype=float)
tic_labels = [f"TIC {str(t)[-6:]}" for t in sorted_vdf['tic_id']]
ml_labels  = list(sorted_vdf['ml_label'])

fig, (ax_h, ax_bar) = plt.subplots(
    1, 2, figsize=(18, max(6, len(sorted_vdf) * 0.6 + 1.5)),
    gridspec_kw={'width_ratios': [3, 1]}
)

# Heatmap
im = ax_h.imshow(score_arr, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax_h, label='Score (0 = bad, 1 = good)', fraction=0.025, pad=0.02)
ax_h.set_xticks(range(len(metrics_map)))
ax_h.set_xticklabels(list(metrics_map.keys()), rotation=30, ha='right', fontsize=10)
ax_h.set_yticks(range(len(tic_labels)))
ax_h.set_yticklabels(
    [f"{t}  [{ml_labels[i].replace('_',' ').title()[:12]}]" for i, t in enumerate(tic_labels)],
    fontsize=9
)
for i in range(score_arr.shape[0]):
    for j in range(score_arr.shape[1]):
        val = score_arr[i, j]
        ax_h.text(j, i, f'{val:.2f}', ha='center', va='center',
                  fontsize=8.5, fontweight='bold',
                  color='black' if 0.25 < val < 0.85 else 'white')
ax_h.set_title('(A) Vetting Scorecard — All Targets\n'
               '(Green = passes diagnostic, Red = fails)',
               fontsize=11, fontweight='bold')

# Overall vetting score bars
overall  = score_arr.mean(axis=1)
bar_cols = [LABEL_COLORS.get(l, '#95a5a6') for l in ml_labels]
ax_bar.barh(range(len(tic_labels)), overall, color=bar_cols,
            edgecolor='white', linewidth=0.5, height=0.7)
ax_bar.axvline(0.5, color='#e74c3c', linestyle='--', lw=1.5, label='Pass threshold (0.5)')
ax_bar.axvline(0.7, color='#2ecc71', linestyle=':', lw=1.5, label='High confidence (0.7)')
for i, v in enumerate(overall):
    ax_bar.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=8.5, fontweight='bold')
ax_bar.set_yticks(range(len(tic_labels)))
ax_bar.set_yticklabels(tic_labels, fontsize=9)
ax_bar.set_xlabel('Mean Vetting Score', fontsize=10)
ax_bar.set_xlim(0, 1.18)
ax_bar.set_title('(B) Overall Vetting Score', fontsize=11, fontweight='bold')
ax_bar.legend(fontsize=8, loc='lower right')
ax_bar.invert_yaxis()

patches_leg = [mpatches.Patch(color=v, label=k.replace('_', ' ').title())
               for k, v in LABEL_COLORS.items() if k in final_df['ml_label'].values]
fig.legend(handles=patches_leg, loc='lower center', ncol=len(patches_leg),
           fontsize=9, bbox_to_anchor=(0.5, -0.06), framealpha=0.9)

plt.suptitle('Figure 5 — Robovetter-Style Vetting Scorecard\n'
             'SNR / Depth / FAP / BIC / ML Confidence / Not-EB diagnostics per target',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'FIGURE5_vetting_scorecard.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print('FIGURE 5 saved!')
n_pass = (score_arr.mean(axis=1) >= 0.5).sum()
print(f'Stars passing vetting (mean score ≥ 0.5): {n_pass} / {len(sorted_vdf)}')

## 7. Print Final Output Summary

In [ ]:
print('=' * 65)
print('  EXOPLANET DETECTION PIPELINE — FINAL RESULTS')
print('=' * 65)
print(f'  Total stars analyzed     : {len(final_df)}')
label_cnt = final_df['ml_label'].value_counts()
for lbl, cnt_v in label_cnt.items():
    print(f'  {lbl.replace("_"," ").title():28s}: {cnt_v}')
print()
print('  Report-quality figures saved to outputs/plots/:')
figs = ['FIGURE1_pipeline_overview.png', 'FIGURE2_classification_dashboard.png',
        'FIGURE3_top6_phasefold.png',    'FIGURE4_parameters.png',
        'FIGURE5_vetting_scorecard.png']
for f in figs:
    path = os.path.join(PLOTS_DIR, f)
    status = '✓' if os.path.exists(path) else '✗ MISSING'
    print(f'    {status}  {f}')
print('=' * 65)
print()
top3 = final_df.nlargest(3, 'snr_display')[
    ['tic_id', 'ml_label', 'ml_confidence', 'period_days', 'depth_ppm', 'snr_display']
]
print('Top 3 candidates by SNR:')
print(top3.to_string(index=False))

---
## All 5 report-quality figures saved in `outputs/plots/`

| Figure | Content | Best for |
|---|---|---|
| FIGURE1_pipeline_overview.png | Raw → detrended → phase-fold → transit zoom | Methodology section |
| FIGURE2_classification_dashboard.png | Pie + SNR bars + Period-Depth space + Confidence dist | Results section |
| FIGURE3_top6_phasefold.png | Top 6 stars phase-folded with binned error bars | Results gallery |
| FIGURE4_parameters.png | Styled results table + planet radii dot plot | Parameter table |
| FIGURE5_vetting_scorecard.png | Heatmap of 6 diagnostic scores per target | Vetting / Discussion |